
# 🤖 🧠 Multi-Agent Engineering Team <br> with Memory & RAG Awareness

This notebook demonstrates how to build a **collaborative AI engineering team** with **(improved and optimized)** memory persistence and RAG implementation. 

**The system combines:**

- multi-agent collaboration  
- persistent memory (short- & long-term)  
- retrieval-augmented generation (RAG)  
- structured workflows & review loops  

**Workflow Architecture:**

**The Project Steps:**

1. Setup llm & prompts
2. Build knowledge & memory
3. Add tools & Utility functions
4. Create agents
5. Build the agents workflow
6. Test & Run the engineering AI agents team 

## Installation & Setup

In [1]:
import importlib.util, subprocess, sys

_pkgs = ["llama-index", "llama-index-llms-openai", "llama-index-vector-stores-chroma", "llama-index-readers-file", "chromadb", "tavily-python"]
_missing = [p for p in _pkgs if importlib.util.find_spec(p.replace("-", "_").split("[")[0]) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _pkgs)

Note: you may need to restart the kernel to use updated packages.


## Step 1 — Load Env Var & Setup LLM (OpenAI)

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # prevents OpenMP crash on macOS when chromadb loads

from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o-mini")

In [3]:
DEV_PROMPT = """
You are a software engineer.
Write clean, working code to implement the requested feature.
Focus on correctness and clarity.
You can search the web for information and record notes when needed.
Hand off to DevAgentB for improvement.
Use:
1. Past issues from memory: {memory_context}
2. Coding standards: {standards_context}
3. Security best practices

Check for:
- bugs
- security risks
- performance issues
- best practices

Suggest improvements if needed.
Hand off to LeadEngineerAgent.
"""

SENIOR_DEV_PROMPT = """
You are a senior software engineer and Refactor Agent.
Your role is to refactor, optimize, and improve existing code.
Use the notes or code provided by the Developer Agent.
Use:
1. Past issues from memory: {memory_context}
2. Coding standards: {standards_context}
3. Security best practices
"""


REVIEW_PROMPT = """
You are a strict reviewer.
Review the code for bugs, security risks, performance, and suggest improvements.

Use:
1. Past issues from memory: {memory_context}
2. Coding standards: {standards_context}
3. Security best practices
"""

LEAD_DEV_PROMPT = """
You are the Lead Engineer.
Your role is to review and validate the work.

Evaluate the following:

- Requirements fulfillment
- Code quality & readability
- Maintainability & architecture
- Security & safety considerations
based on {standards_context}

PROCESS:
1. First, delegate the review to DevAgentA.
2. Analyze DevAgentA’s findings.
3. Make the final decision.

Take into account past issues and tasks requests with {memory_context}

OUTPUT FORMAT (MANDATORY): return ONLY one of the following:
    ✅ APPROVE  
    → if everything is acceptable and ready for production.

    ❌ REJECT  
    → if major issues exist that prevent acceptance.

    ⚠️ NEEDS IMPROVEMENT  
    → if mostly acceptable but requires revisions or enhancements.

After the decision, include a concise justification and actionable feedback if applicable.
"""

## Step 2 - RAG for Coding Standards 

Retrieve : 
- internal coding standards
- security practices
- architecture rules
- performance guidelines

In [4]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

docs = SimpleDirectoryReader("standards_docs").load_data()
index = VectorStoreIndex.from_documents(docs)

retriever = index.as_retriever(similarity_top_k=2)

def get_standards_context(query: str):
    nodes = retriever.retrieve(query or "python best practices security")
    return "\n".join(n.node.text for n in nodes)

2026-09-19 19:26:07,884 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


## Step 3 - Creating Memory Storage 
- A memory block that stores and retrieves batches of chat messages from a vector database.
- Retrieve past memories during coding tasks and reviews

**Documentation**
[Chroma Vector Store](https://developers.llamaindex.ai/python/framework/integrations/vector_stores/chroma_metadata_filter/)

In [5]:
from llama_index.core import VectorStoreIndex, Document
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.storage.storage_context import StorageContext
import chromadb

# persistent DB
client = chromadb.Client()
collection = client.get_or_create_collection("engineering_memory")

vector_store = ChromaVectorStore(chroma_collection=collection)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

memory_index = VectorStoreIndex([], storage_context=storage_context)

memory_retriever = memory_index.as_retriever(similarity_top_k=3)

# store past conversations and review feedback in memory
def store_review_feedback(text: str, doc_id: str = "latest_feedback"):
    """Store a review feedback message in memory_index as a Document."""
    doc = Document(text=text, doc_id=doc_id)
    memory_index.insert(doc)
    
def retrieve_past_issues(query: str = "past issues and feedback"):
    return memory_retriever.retrieve(query)

def retrieve_print_memory(query: str = "recent issues and feedback"):
    memories = memory_retriever.retrieve(query)
    return "\n".join([n.node.text for n in memories])
 

2026-09-19 19:26:08,263 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


## Memory Pruning & Summarization (Optimization)

- the process of selectively removing low-value information while compressing important context into concise summaries. 
- helps AI systems manage limited context windows, reduce storage costs, and maintain long-term coherence by preserving what matters most while discarding redundancy or outdated data.

In [6]:
async def summarize_memory(llm, keep_recent=5):
    messages = retrieve_past_issues("recent issues and feedback")  
    if len(messages) <= keep_recent:
        return

    old_messages = messages[:-keep_recent]
    recent_messages = messages[-keep_recent:]

    old_text = "\n".join([f"{m.role}: {m.content}" for m in old_messages])
    summary_prompt = f"Summarize these older messages concisely, keeping key decisions:\n{old_text}"

    summary_response = await llm.invoke({"messages":[{"role":"user","content":summary_prompt}]})
    summary_text = summary_response["messages"][-1].content

    # remove old messages and store summary
    for _ in old_messages:
        memory_index.delete()
    memory_index.insert(summary_text)

async def prune_memory_if_needed(llm, max_tokens=4000, keep_recent=5):
    messages = retrieve_past_issues()
    current_tokens = sum(len(m.node.text)//4 for m in messages)
    if current_tokens > max_tokens:
        print(f"⚠️ Memory token count {current_tokens} exceeds {max_tokens}. Summarizing...")
        await summarize_memory(llm, keep_recent)
    else:
        print(f"✅ Memory token count {current_tokens} within limit.")

## Step 4 - Add Tools & Functions

🔄 Workflow Flow

In [7]:
from tavily import AsyncTavilyClient
from llama_index.core.workflow import Context

def write_code(user_request: str | None = "recent issues and feedback"):
    memory_context = retrieve_print_memory(user_request)
    return DEV_PROMPT.format(
        memory_context=memory_context,
    )
    
def refactor_code(code: str):
    memory_context = retrieve_print_memory(code)

    return SENIOR_DEV_PROMPT.format(
        memory_context=memory_context,
    )

def review_code(code: str):
    memory_context = retrieve_print_memory(code)
    standards_context = get_standards_context()

    prompt = REVIEW_PROMPT.format(
        memory_context=memory_context,
        standards_context=standards_context
    )

    return prompt + "\n\nCode:\n" + code

def build_final_review_prompt(code: str):
    memory_context = retrieve_print_memory(code)

    final_review = LEAD_DEV_PROMPT.format(
        memory_context=memory_context,
    )
    return final_review


async def search_web(query: str) -> str:
    """Allow the agent to search the web for information."""
    client = AsyncTavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    return str(await client.search(query))


async def record_notes(ctx: Context, notes: str, notes_title: str) -> str:
    """Store research or implementation notes."""

    code_output = await write_code(notes)
    
    await ctx.store.set_state({
        "state": {
            "research_notes": {},
            "review_report_content": "",
            "refactored_notes"
            "final_review_report": ""   
        }
    })

    async with ctx.store.edit_state() as state:
        state["research_notes"][notes_title] = code_output
    return f"Notes '{notes_title}' recorded."

async def refactor_code(ctx: Context, notes_title: str) -> str:
    """
    Refactor code based on developer notes or previous research notes.
    Updates the workflow state with refactored_notes.
    """
    # Retrieve the original notes from ctx
    async with ctx.store.edit_state() as ctx_state:
        original_notes = ctx_state["state"].get("research_notes", {}).get(notes_title, "")
    
    if not original_notes:
        return f"No notes found for title '{notes_title}' to refactor."
    
    # Refactor using the agent
    refactored_output = await refactor_code(original_notes)
    
    # Save to workflow state
    async with ctx.store.edit_state() as ctx_state:
        ctx_state["state"]["refactored_notes"][notes_title] = refactored_output
    
    return f"Refactored notes saved for '{notes_title}'."

async def write_review_report(ctx: Context) -> str:
    """Generate a report from recorded notes."""

    async with ctx.store.edit_state() as state:
        refactored_notes = state.get("refactored_refactored_notes", {})

    if not refactored_notes:
        return "No refactored_notes available to build report."

    combined_notes = "\n\n".join(refactored_notes.values())
    review_output = await review_code(combined_notes)

    async with ctx.store.edit_state() as state:
        state["state"]["review_report_content"] = review_output

    return "Review Report written."


async def review_report(ctx: Context) -> str:
    """Review the generated report."""

    async with ctx.store.edit_state() as state:
        review_report = state.get("review_report_content")

    if not review_report:
        return "No report available for review."

    final_review_output = await build_final_review_prompt(review_report) + "\n\nReport:\n" + review_report

    async with ctx.store.edit_state() as state:
        state["state"]["final_review_report"] = final_review_output

    return "Report reviewed with final review."


## Step 5 — Create the Engineering Agents
[LlamaIndex Agents Workflow](https://developers.llamaindex.ai/python/llamaagents/workflows/)

In [8]:
from llama_index.core.agent.workflow import FunctionAgent


dev_agent = FunctionAgent(
    name="Developer",
    description="Writes code",
    system_prompt=DEV_PROMPT,
    llm=llm,
    tools=[search_web, record_notes],
    can_handoff_to=["Senior_Developer"],
    streaming=True,
)

senior_dev_agent = FunctionAgent(
    name="Senior_Developer",
    description="Refactor code",
    system_prompt=SENIOR_DEV_PROMPT,
    llm=llm,
    tools=[refactor_code, search_web, record_notes],
    can_handoff_to=["Reviewer"],
    streaming=True,
)

review_agent = FunctionAgent(
    name="Reviewer",
    description="Reviews code",
    system_prompt=REVIEW_PROMPT,
    llm=llm,
    tools=[write_review_report],
    can_handoff_to=["Lead"],
    streaming=True,
)

lead_agent = FunctionAgent(
    name="Lead",
    description="Approves code and review",
    system_prompt=LEAD_DEV_PROMPT,
    llm=llm,
    tools=[review_report],
    streaming=True,
)

## Creating the Agent Workflow

In [9]:
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.workflow import Context

workflow = AgentWorkflow(
    agents=[dev_agent, senior_dev_agent,review_agent, lead_agent],
    root_agent="Developer",
)
ctx = Context(workflow)

In [10]:
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCallResult,
    AgentStream,
)

async def run_workflow(user_msg: str | None = None):
    print("\n🚀 Engineering Team Ready\n")

    while True:
        task = input("\nTask (exit to quit): ")

        if task.lower() == "exit":
            break
        
        # use user_msg if provided, otherwise use task
        user_request = user_msg or task
        
        handler = workflow.run(user_msg=user_request)

        print("⚙️ Task in Progress ..\n")

        async for event in handler.stream_events():

            # ✅ Stream tokens (LLM typing)
            if isinstance(event, AgentStream):
                if event.delta:
                    print(event.delta, end="", flush=True)

            # ✅ Agent started
            elif isinstance(event, AgentInput):
                print(f"🤖 {event.current_agent_name} working...\n")

            # ✅ Tool results
            elif isinstance(event, ToolCallResult):
                print(f"\n🔧 Tool Used: {event.tool_name}")
                print(f"→ Result: {event.tool_output}\n")

            # ✅ Agent finished response
            elif isinstance(event, AgentOutput):
                if event.response:
                    agent_name = getattr(event, "current_agent_name", "Unknown")
                    print("\n" + "-" * 40)
                    print(f"✅ Agent finished: {agent_name}")
                    print("-" * 40)
                    print(event.response)
                    # store per agent           
                    store_review_feedback(f"{agent_name} output:\n{event.response}")

        final = await handler

        print("\n" + "="*60)
        print("✅ FINAL RESULT")
        print("="*60)
        print(str(final))      

        store_review_feedback(str(await handler))

        prune_memory_if_needed(llm)



## For Testing Memory

In [11]:
# for testing memory - Store feedback
store_review_feedback("Bug in endpoint upload causing crashes under load.")
store_review_feedback("Optimized database queries to reduce latency.")

# Print memory
def print_memory():
    """Print all messages in memory, handling both string and Node objects."""
    history = retrieve_past_issues()  # returns list of strings or Node objects
    print(f"Total messages in memory: {len(history)}")
    
    for msg in history:
        # Check if it's a Node object or a string
        if hasattr(msg, "node") and hasattr(msg.node, "text"):
            print(msg.node.text)
        else:
            print(msg)  # fallback: msg is already a string

print_memory()

2026-09-19 19:26:08,575 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-09-19 19:26:08,756 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-09-19 19:26:09,010 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Total messages in memory: 2
Optimized database queries to reduce latency.
Bug in endpoint upload causing crashes under load.


### -- Mock Test for Memory Pruning & Summarization --

1. Mock user and agent messages to simulate memory growth.
2. Calls a mock prune/summarize function that keeps only the latest messages (or could call your real summarization LLM).
3. Prints memory before and after pruning, so you can visually verify that old messages are removed or summarized.

In [12]:
# --- Mock Test for Memory Pruning & Summarization  ---

from llama_index.core.llms import ChatMessage

from collections import deque

# --- Simple in-memory mock memory for testing ---
class MockMemory:
    def __init__(self):
        self.messages = deque() 

    def put_messages(self, messages):
        self.messages.extend(messages)


    def get(self):
        return list(self.messages)    

    def clear(self):
        self.messages.clear()

    def __len__(self):
        return len(self.messages)

memory = MockMemory()


# --- Helper functions ---
def store_memory(msg: dict):
    """Store a message in memory."""
    memory.put_messages([ChatMessage(role=msg["role"], content=msg["content"])])

def format_memory():
    return "\n".join([f"{m.role}: {m.content}" for m in memory.get()])

# --- Mock summarize memory function ---
def mock_summarize_memory(keep_last_n=5):
    """Simulate summarizing older messages by keeping only last N messages."""

    chat_history = memory.get()
    if len(chat_history) <= keep_last_n:
        return  # nothing to summarize

    # Keep only the last N messages
    summarized = chat_history[-keep_last_n:]
    memory.clear()
    memory.put_messages(summarized)
    print(f"🔹 Memory summarized, kept last {keep_last_n} messages.")


def mock_prune_memory_if_needed(max_messages=5):
    """Prune memory if it exceeds max_messages."""

    if len(memory.get()) > max_messages:
        mock_summarize_memory(keep_last_n=max_messages)

# --- Add test messages ---

def add_mock_test_messages(num_messages=10):
    """Add a series of test messages to memory."""
    for i in range(num_messages):
        store_memory({"role": "user", "content": f"User message {i}"})
        store_memory({"role": "assistant", "content": f"Assistant response {i}"})


Run and mock memory growth and pruning 

In [13]:
# Seed mock memory so the print output is visible
add_mock_test_messages(num_messages=10)

# --- Check memory before pruning ---
print("📄 Memory BEFORE pruning:")
print(format_memory())


# --- Run pruning test ---
mock_prune_memory_if_needed(max_messages=5)

# --- Check memory after pruning ---
print("\n📄 Memory AFTER pruning/summarization:")
print(format_memory())


📄 Memory BEFORE pruning:
MessageRole.USER: User message 0
MessageRole.ASSISTANT: Assistant response 0
MessageRole.USER: User message 1
MessageRole.ASSISTANT: Assistant response 1
MessageRole.USER: User message 2
MessageRole.ASSISTANT: Assistant response 2
MessageRole.USER: User message 3
MessageRole.ASSISTANT: Assistant response 3
MessageRole.USER: User message 4
MessageRole.ASSISTANT: Assistant response 4
MessageRole.USER: User message 5
MessageRole.ASSISTANT: Assistant response 5
MessageRole.USER: User message 6
MessageRole.ASSISTANT: Assistant response 6
MessageRole.USER: User message 7
MessageRole.ASSISTANT: Assistant response 7
MessageRole.USER: User message 8
MessageRole.ASSISTANT: Assistant response 8
MessageRole.USER: User message 9
MessageRole.ASSISTANT: Assistant response 9
🔹 Memory summarized, kept last 5 messages.

📄 Memory AFTER pruning/summarization:
MessageRole.ASSISTANT: Assistant response 7
MessageRole.USER: User message 8
MessageRole.ASSISTANT: Assistant response 8
Me

### -- Test (Vector Storage )for Memory Pruning & Summarization --

In [14]:
import time

def add_test_messages(num_messages=10, prefix="test_msg", target_collection=None):
    """Add test messages to the collection with specific metadata for pruning tests."""
    coll = target_collection or collection
    data = coll.get(include=["metadatas"])
    existing_ids = data.get("ids", []) or []
    existing_meta = data.get("metadatas", []) or []
    to_delete = []
    for idx, doc_id in enumerate(existing_ids):
        md = existing_meta[idx] if idx < len(existing_meta) and existing_meta[idx] else {}
        if md.get("source") == "prune_test" and md.get("group") == prefix:
            to_delete.append(doc_id)
    if to_delete:
        coll.delete(ids=to_delete)

    for i in range(num_messages):
        doc = Document(
            text=f"{prefix} content {i}",
            doc_id=f"{prefix}_{i}",
            metadata={"source": "prune_test", "group": prefix, "seq": i, "ts": time.time_ns()},

        )
        memory_index.insert(doc)

    print(f"Added {num_messages} test messages with group '{prefix}'.")



def prune_memory(last_n=5, prefix="test_msg", target_collection=None):
    """Prune test messages in the collection, keeping only the last N based on 'seq' metadata."""
    coll = target_collection or collection

    # Retrieve all docs and their metadatas
    data = coll.get(include=["metadatas"])
    all_ids = data.get("ids", []) or []
    all_meta = data.get("metadatas", []) or []

    # Filter test docs by metadata and pair with seq for sorting
    test_items = []
    for idx, doc_id in enumerate(all_ids):
        md = all_meta[idx] if idx < len(all_meta) and all_meta[idx] else {}
        if md.get("source") == "prune_test" and md.get("group") == prefix:
            seq = md.get("seq", -1)
            test_items.append((seq, doc_id))



    if len(test_items) <= last_n:
        print(f"No pruning needed, only {len(test_items)} test messages present.")
        return


    test_items.sort(key=lambda x: x[0])
    ids_to_delete = [doc_id for _, doc_id in test_items[:-last_n]]
    coll.delete(ids=ids_to_delete)
    print(f"🧹 Pruned {len(ids_to_delete)} test messages, kept last {last_n}.")



def get_test_messages(prefix="test_msg", target_collection=None):
    """Retrieve test messages from the collection, sorted by their 'seq' metadata."""
    coll = target_collection or collection
    data = coll.get(include=["documents", "metadatas"])
    all_ids = data.get("ids", []) or []
    all_docs = data.get("documents", []) or []
    all_meta = data.get("metadatas", []) or []
    rows = []

    for idx, doc_id in enumerate(all_ids):
        md = all_meta[idx] if idx < len(all_meta) and all_meta[idx] else {}
        if md.get("source") == "prune_test" and md.get("group") == prefix:
            text = all_docs[idx] if idx < len(all_docs) else ""
            seq = md.get("seq", -1)
            rows.append((seq, doc_id, text, md))

    rows.sort(key=lambda x: x[0])
    return rows



def print_test_messages(prefix="test_msg", target_collection=None, limit=None):
    """Helper to print test messages in a readable format."""
    rows = get_test_messages(prefix=prefix, target_collection=target_collection)
    print(f"Total test messages: {len(rows)}")
    if limit is not None:
        rows = rows[:limit]
    for seq, doc_id, text, md in rows:
        print(f"seq={seq} | {doc_id}: {text} | metadata: {md}")


Test storage and pruning with Vector Store

In [15]:
prefix = "test_msg"

add_test_messages(num_messages=30, prefix=prefix, target_collection=collection)

before_rows = get_test_messages(prefix=prefix, target_collection=collection)

print("📄 BEFORE pruning")
print(f"Count: {len(before_rows)}")
print("First 3:")
for row in before_rows[:3]:
    print(row[2])
print("Last 3:")
for row in before_rows[-3:]:
    print(row[2])


prune_memory(last_n=5, prefix=prefix, target_collection=collection)


after_rows = get_test_messages(prefix=prefix, target_collection=collection)
print("\n📄 AFTER pruning")
print(f"Count: {len(after_rows)}")
for row in after_rows:
    print(row[2])


assert len(after_rows) == 5, f"Expected 5 rows after prune, got {len(after_rows)}"
print("\n✅ Pruning test passed (kept 5).")


2026-09-19 19:26:09,218 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-09-19 19:26:09,399 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-09-19 19:26:09,606 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-09-19 19:26:09,868 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-09-19 19:26:10,053 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


...

2026-09-19 19:26:14,587 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-09-19 19:26:14,783 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-09-19 19:26:14,986 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2026-09-19 19:26:15,146 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Added 30 test messages with group 'test_msg'.
📄 BEFORE pruning
Count: 30
First 3:
test_msg content 0
test_msg content 1
test_msg content 2
Last 3:
test_msg content 27
test_msg content 28
test_msg content 29
🧹 Pruned 25 test messages, kept last 5.

📄 AFTER pruning
Count: 5
test_msg content 25
test_msg content 26
test_msg content 27
test_msg content 28
test_msg content 29

✅ Pruning test passed (kept 5).


## Step 6 — Run the Engineering Team (Try it out!)

**Coding Tasks Examples**

1. Create a REST API endpoint for user registration
- Refactor the REST API endpoint for user registration, with validation logic into a separate service, error handling with centralized middleware
- Update the REST API endpoint for user registration : add email verification step, include rate limiting for security

2. Create a Python function to validate email addresses
- update to add type hints and docstrings and Support internationalized email addresses

In [16]:
# await run_workflow()